Fanny BADOULES  
Maëlys HANOIRE  
Diane VERBECQ  


# Projet SDP

## Question 1

Formuler un programme d’optimisation linéaire qui calcule une explication de
type (1-1) de la comparaison x ≻ y si elle existe, et retourne un certificat de non-existence
dans le cas contraire. Impl´ementez cette formulation en utilisant un solveur d’optimisation.

In [9]:
from gurobipy import Model, GRB, quicksum

# Données de l'exemple x (Xavier) vs y (Yvonne)
courses = ["A","B","C","D","E","F","G"]

x = {"A":85, "B":81, "C":71, "D":69, "E":75, "F":81, "G":88}
y = {"A":81, "B":81, "C":75, "D":63, "E":67, "F":88, "G":95}

w = {"A":8, "B":7, "C":7, "D":6, "E":6, "F":5, "G":6}


In [ ]:
# Contributions
delta = {k: w[k]*(x[k]-y[k]) for k in courses}

pros    = [k for k in courses if delta[k] > 0]
cons    = [k for k in courses if delta[k] < 0]
neutral = [k for k in courses if delta[k] == 0]

print("Δ:", delta)
print("pros(x,y):", pros)
print("cons(x,y):", cons)
print("neutral(x,y):", neutral)

# Ensemble T des paires admissibles (1-1)
T = [(p,c) for p in pros for c in cons if delta[p] + delta[c] > 0]
print("\nNombre de trade-offs admissibles |T| =", len(T))
print("T =", T)


Δ: {'A': 32, 'B': 0, 'C': -28, 'D': 36, 'E': 48, 'F': -35, 'G': -42}
pros(x,y): ['A', 'D', 'E']
cons(x,y): ['C', 'F', 'G']
neutral(x,y): ['B']

Nombre de trade-offs admissibles |T| = 6
T = [('A', 'C'), ('D', 'C'), ('D', 'F'), ('E', 'C'), ('E', 'F'), ('E', 'G')]


On calcule la contribution de chaque matière à la comparaison x>y, puis on classe les matières en favorables (pros), défavorables (cons) ou neutres.

On en déduit l’ensemble T des trade-offs (1-1) possibles, où une matière favorable compense une matière défavorable avec un gain global positif.

In [11]:
m = Model("Explanation_1-1")

# Variables binaires z[p,c] = 1 si on choisit le trade-off (p,c)
z = m.addVars(T, vtype=GRB.BINARY, name="z")

# Chaque critère "contre" doit être couvert exactement une fois
for c in cons:
    m.addConstr(quicksum(z[p,c] for p in pros if (p,c) in z) == 1, name=f"cover_{c}")

# Chaque critère "pour" utilisé au plus une fois (disjonction côté pros)
for p in pros:
    m.addConstr(quicksum(z[p,c] for c in cons if (p,c) in z) <= 1, name=f"use_{p}_at_most_once")

# Objectif : minimiser la longueur (nombre de paires)
m.setObjective(quicksum(z[p,c] for (p,c) in T), GRB.MINIMIZE)

# Mode silencieux
m.params.OutputFlag = 0

m.optimize()


On crée le modèle d’optimisation et des variables binaires qui indiquent quels trade-offs (1-1) sont sélectionnés pour expliquer
x>y.

Les contraintes imposent que chaque matière défavorable soit compensée exactement une fois, et l’objectif minimise le nombre de trade-offs utilisés pour obtenir l’explication la plus simple.

In [ ]:
status = m.Status

if status == GRB.OPTIMAL:
    E = [(p,c) for (p,c) in T if z[p,c].X > 0.5]
    print("Explication (1-1) trouvée")
    print("Longueur l =", len(E))
    print("E =", E)

    covered_cons = sorted([c for (_,c) in E])
    print("Couverts (cons) =", covered_cons)

    print("\nInterprétation :")
    for p,c in E:
        print(f"- l'avantage en {p} (Δ={delta[p]}) compense le désavantage en {c} (Δ={delta[c]}), somme = {delta[p]+delta[c]}>0")

elif status == GRB.INFEASIBLE:
    print("Aucune explication (1-1) n'existe : modèle INFaisable.")


else:
    print("Statut solveur :", status)


Explication (1-1) trouvée
Longueur l = 3
E = [('A', 'C'), ('D', 'F'), ('E', 'G')]
Couverts (cons) = ['C', 'F', 'G']

Interprétation :
- l'avantage en A (Δ=32) compense le désavantage en C (Δ=-28), somme = 4>0
- l'avantage en D (Δ=36) compense le désavantage en F (Δ=-35), somme = 1>0
- l'avantage en E (Δ=48) compense le désavantage en G (Δ=-42), somme = 6>0


Le solveur trouve une explication (1-1) de longueur 3 : chaque matière défavorable à x>y (C, F, G) est compensée par une matière favorable (A, D, E) avec un bilan positif.
Concrètement, on justifie x>y par trois compensations : (A) bat (C), (D) bat (F) et (E) bat (G) (les sommes (4), (1) et (6) sont toutes (>0)).


## Question 2
Formuler un programme d’optimisation lin´eaire qui calcule une explication de
type (1-m) de la comparaison x ≻ y si elle existe, et retourne un certificat de non-existence
dans le cas contraire. Impl´ementez cette formulation en utilisant un solveur d’optimisation.